# Paso 7 — Motor de Recomendaciones — FinanceAI

Este notebook implementa el motor de recomendaciones financieras que completa el contrato de la API descrito en el brief (`POST /analisis-financiero`). A diferencia de los dos clasificadores anteriores, **este motor no usa Machine Learning**: son reglas simples y explicables sobre los resultados ya calculados (`resumen_gastos`, `ratio_gasto_ingreso`, `nivel_endeudamiento`, `frecuencia_ahorro`, `perfil_financiero`).

Con esto, el pipeline de Ciencia de Datos queda completo de extremo a extremo:

```
transacciones crudas
   -> clasificador de gastos (Paso 5)
   -> features agregadas
   -> clasificador de perfil (Paso 6)
   -> motor de recomendaciones (este notebook)
   -> JSON de salida identico al del brief
```

**Contenido:**
1. Carga de artefactos (ambos modelos entrenados + transformadores)
2. Reglas de recomendacion (umbrales documentados y su justificacion)
3. Funcion de analisis financiero completo (integra los 3 componentes)
4. Prueba con ejemplos reales, incluyendo el ejemplo exacto del brief
5. Serializacion de las reglas para el equipo de backend

In [1]:
import re
import json
import unicodedata

import pandas as pd
import joblib

RANDOM_STATE = 42

## 1. Carga de artefactos

In [2]:
modelo_gastos = joblib.load("artefactos/modelo_clasificador_gastos.pkl")
vectorizador_tfidf = joblib.load("artefactos/tfidf_vectorizer.pkl")
modelo_perfil = joblib.load("artefactos/modelo_perfil_financiero.pkl")
scaler_perfil = joblib.load("artefactos/scaler_perfil.pkl")
encoder_ahorro = joblib.load("artefactos/encoder_frecuencia_ahorro.pkl")

with open("artefactos/metadata_modelo_perfil.json", encoding="utf-8") as f:
    metadata_perfil = json.load(f)

columnas_features = metadata_perfil["columnas_features"]
columnas_numericas_continuas = metadata_perfil["columnas_numericas_continuas"]
requiere_escalado = metadata_perfil["requiere_escalado"]

with open("artefactos/metadata_modelo_gastos.json", encoding="utf-8") as f:
    metadata_gastos = json.load(f)

print(f"Modelo de gastos cargado: {metadata_gastos['modelo']} (accuracy test: {metadata_gastos['accuracy_test']})")
print(f"Modelo de perfil cargado: {metadata_perfil['modelo']} (accuracy test: {metadata_perfil['accuracy_test']})")
print(f"Requiere escalado de features: {requiere_escalado}")

Modelo de gastos cargado: Linear SVC (accuracy test: 0.8984)
Modelo de perfil cargado: Gradient Boosting (accuracy test: 0.9375)
Requiere escalado de features: False


## 2. Funciones auxiliares (mismas del Paso 6)

Se redefinen aqui para que este notebook sea autocontenido y pueda ejecutarse de forma independiente.

In [3]:
CIUDADES_RUIDO = [
    "ciudad de mexico", "bogota", "medellin", "barranquilla", "cartagena",
    "guadalajara", "monterrey", "queretaro", "cucuta", "cali", "tijuana", "puebla",
]

def quitar_tildes(texto):
    texto_norm = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto_norm if not unicodedata.combining(c))

def limpiar_descripcion(texto):
    texto = texto.lower()
    texto = quitar_tildes(texto)
    texto = re.sub(r"ref\d+", " ", texto)
    texto = re.sub(r"#\d+", " ", texto)
    for ciudad in CIUDADES_RUIDO:
        texto = texto.replace(ciudad, " ")
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

CATEGORIAS_BASE = ["alimentacion", "transporte", "salud", "vivienda", "educacion", "ocio", "servicios", "otros"]


def construir_features_usuario(ingreso_mensual, nivel_endeudamiento, frecuencia_ahorro, transacciones):
    gasto_por_categoria = {c: 0.0 for c in CATEGORIAS_BASE}

    for t in transacciones:
        texto_limpio = limpiar_descripcion(t["descripcion"])
        vector = vectorizador_tfidf.transform([texto_limpio])
        categoria_predicha = modelo_gastos.predict(vector)[0].lower()
        gasto_por_categoria[categoria_predicha] = gasto_por_categoria.get(categoria_predicha, 0.0) + t["valor"]

    gasto_total = sum(gasto_por_categoria.values())
    num_transacciones = len(transacciones)
    ratio_gasto_ingreso = gasto_total / ingreso_mensual if ingreso_mensual else 0.0
    gasto_promedio_transaccion = gasto_total / num_transacciones if num_transacciones else 0.0
    categoria_dominante = max(gasto_por_categoria, key=gasto_por_categoria.get)

    fila = {
        "ingreso_mensual": ingreso_mensual,
        "nivel_endeudamiento": nivel_endeudamiento,
        "num_transacciones": num_transacciones,
        "gasto_total": gasto_total,
        "ratio_gasto_ingreso": ratio_gasto_ingreso,
        "gasto_promedio_transaccion": gasto_promedio_transaccion,
    }
    for c in CATEGORIAS_BASE:
        fila[f"gasto_{c}"] = gasto_por_categoria[c]
        fila[f"pct_{c}"] = (gasto_por_categoria[c] / gasto_total) if gasto_total else 0.0
        fila[f"dominante_{c}"] = 1 if c == categoria_dominante else 0
    fila["frecuencia_ahorro_cod"] = float(
        encoder_ahorro.transform(pd.DataFrame([[frecuencia_ahorro]], columns=["frecuencia_ahorro"]))[0][0]
    )

    df_fila = pd.DataFrame([fila]).reindex(columns=columnas_features, fill_value=0)
    return df_fila, gasto_por_categoria


def predecir_perfil(df_fila):
    if requiere_escalado:
        df_fila_modelo = df_fila.copy()
        df_fila_modelo[columnas_numericas_continuas] = scaler_perfil.transform(df_fila[columnas_numericas_continuas])
    else:
        df_fila_modelo = df_fila

    perfil_predicho = modelo_perfil.predict(df_fila_modelo)[0]
    probabilidades = modelo_perfil.predict_proba(df_fila_modelo)[0]
    clases = list(modelo_perfil.classes_)
    probabilidad = float(probabilidades[clases.index(perfil_predicho)])
    return perfil_predicho, probabilidad

## 3. Reglas de recomendacion

Las reglas se basan en tres senales, cada una con un umbral documentado:

| Senal | Umbral | Justificacion |
|---|---|---|
| Gasto por categoria / ingreso | Vivienda 35%, Alimentacion 25%, Transporte 15%, Ocio 10%, Servicios 10%, Otros 15% | Referencias habituales de presupuesto personal (regla 50/30/20 adaptada por categoria). Salud y Educacion **no** se limitan: gastar mas ahi no es necesariamente un problema. |
| Gasto total / ingreso (`ratio_gasto_ingreso`) | >90% = alerta alta, >70% = alerta media | Si casi todo el ingreso se va en gastos, no queda margen para imprevistos. |
| Nivel de endeudamiento | >50% = alerta alta, >30% = alerta media | Umbrales tipicos usados por asesores financieros para senalar sobreendeudamiento. |
| Frecuencia de ahorro | "Baja" = alerta media, "Media" = sugerencia leve | El ahorro es el indicador con mayor correlacion con el perfil "Saludable" (visto en el EDA). |

Cada regla que se activa genera una recomendacion con un nivel de **severidad** (3 = alta, 2 = media, 1 = informativa). Las recomendaciones se ordenan por severidad y se limitan a un maximo (`max_recomendaciones`), para no saturar al usuario. El mensaje general segun el `perfil_financiero` siempre se incluye primero.

In [4]:
REGLAS_RECOMENDACIONES = {
    "umbral_pct_categoria": {
        "vivienda": 0.35,
        "alimentacion": 0.25,
        "transporte": 0.15,
        "ocio": 0.10,
        "servicios": 0.10,
        "otros": 0.15,
    },
    "umbral_ratio_alto": 0.90,
    "umbral_ratio_medio": 0.70,
    "umbral_endeudamiento_alto": 50,
    "umbral_endeudamiento_medio": 30,
    "max_recomendaciones": 4,
}

NOMBRES_CATEGORIA = {
    "vivienda": "vivienda",
    "alimentacion": "alimentacion",
    "transporte": "transporte",
    "ocio": "ocio y entretenimiento",
    "servicios": "servicios (facturas, internet, telefonia)",
    "otros": "gastos varios",
}

MENSAJES_PERFIL = {
    "Saludable": "Tu perfil financiero es saludable. Manten tus habitos actuales de gasto y ahorro.",
    "En observacion": "Tu perfil financiero esta en observacion. Pequenos ajustes en tus gastos o en tu frecuencia de ahorro pueden mejorar tu situacion.",
    "En riesgo": "Tu perfil financiero esta en riesgo. Te recomendamos revisar a fondo tus gastos y buscar reducir tu endeudamiento lo antes posible.",
}


def generar_recomendaciones(perfil_financiero, gasto_por_categoria, ingreso_mensual,
                             ratio_gasto_ingreso, nivel_endeudamiento, frecuencia_ahorro,
                             reglas=REGLAS_RECOMENDACIONES):
    candidatas = []

    for categoria, umbral in reglas["umbral_pct_categoria"].items():
        gasto_categoria = gasto_por_categoria.get(categoria, 0.0)
        pct_ingreso = gasto_categoria / ingreso_mensual if ingreso_mensual else 0.0
        if pct_ingreso > umbral:
            severidad = 2 if pct_ingreso > umbral * 1.3 else 1
            nombre = NOMBRES_CATEGORIA.get(categoria, categoria)
            candidatas.append((
                severidad,
                f"Tu gasto en {nombre} representa el {pct_ingreso * 100:.0f}% de tu ingreso mensual "
                f"(recomendado: hasta {umbral * 100:.0f}%); considera reducirlo o revisar estos gastos.",
            ))

    if ratio_gasto_ingreso > reglas["umbral_ratio_alto"]:
        candidatas.append((3, (
            f"Tus gastos totales representan el {ratio_gasto_ingreso * 100:.0f}% de tu ingreso mensual; "
            "estas gastando casi todo (o mas de) lo que ganas. Revisa tu presupuesto con urgencia."
        )))
    elif ratio_gasto_ingreso > reglas["umbral_ratio_medio"]:
        candidatas.append((2, (
            f"Tus gastos totales representan el {ratio_gasto_ingreso * 100:.0f}% de tu ingreso mensual; "
            "te queda poco margen. Vigila tus gastos recurrentes."
        )))

    if nivel_endeudamiento > reglas["umbral_endeudamiento_alto"]:
        candidatas.append((3, (
            f"Tu nivel de endeudamiento ({nivel_endeudamiento:.0f}%) es alto; prioriza reducir el uso "
            "de credito y evita adquirir nuevas deudas."
        )))
    elif nivel_endeudamiento > reglas["umbral_endeudamiento_medio"]:
        candidatas.append((1, (
            f"Tu nivel de endeudamiento ({nivel_endeudamiento:.0f}%) esta en un rango moderado; "
            "evita aumentarlo en los proximos meses."
        )))

    if frecuencia_ahorro == "Baja":
        candidatas.append((2, (
            "Tu frecuencia de ahorro es baja; intenta destinar una porcion fija de tu ingreso "
            "al ahorro cada mes, aunque sea pequena."
        )))
    elif frecuencia_ahorro == "Media":
        candidatas.append((1, "Podrias aumentar tu frecuencia de ahorro para fortalecer tu reserva financiera."))

    candidatas.sort(key=lambda x: x[0], reverse=True)
    recomendaciones_especificas = [mensaje for _, mensaje in candidatas[: reglas["max_recomendaciones"] - 1]]

    recomendaciones = [MENSAJES_PERFIL.get(perfil_financiero, "Revisa tu situacion financiera con un asesor.")]
    recomendaciones.extend(recomendaciones_especificas)

    return recomendaciones[: reglas["max_recomendaciones"]]

## 4. Funcion de analisis financiero completo

In [5]:
def analisis_financiero_completo(ingreso_mensual, nivel_endeudamiento, frecuencia_ahorro, transacciones):
    df_fila, gasto_por_categoria = construir_features_usuario(
        ingreso_mensual, nivel_endeudamiento, frecuencia_ahorro, transacciones
    )
    perfil_predicho, probabilidad = predecir_perfil(df_fila)

    gasto_total = sum(gasto_por_categoria.values())
    ratio_gasto_ingreso = gasto_total / ingreso_mensual if ingreso_mensual else 0.0

    recomendaciones = generar_recomendaciones(
        perfil_predicho, gasto_por_categoria, ingreso_mensual,
        ratio_gasto_ingreso, nivel_endeudamiento, frecuencia_ahorro,
    )

    resumen_gastos = {k: round(v, 2) for k, v in gasto_por_categoria.items() if v > 0}

    return {
        "perfil_financiero": perfil_predicho,
        "probabilidad": round(probabilidad, 4),
        "resumen_gastos": resumen_gastos,
        "recomendaciones": recomendaciones,
    }

## 5. Prueba con ejemplos reales

Probamos con el **ejemplo textual exacto del brief** (`POST /analisis-financiero`) y dos casos adicionales en los extremos del espectro de riesgo, para verificar que las recomendaciones cambian de forma coherente segun la situacion del usuario.

In [6]:
ejemplos = [
    {
        "nombre": "Ejemplo del brief",
        "ingreso_mensual": 4500, "nivel_endeudamiento": 25, "frecuencia_ahorro": "Media",
        "transacciones": [
            {"descripcion": "Supermercado", "valor": 420},
            {"descripcion": "Combustible", "valor": 300},
            {"descripcion": "Streaming", "valor": 40},
        ],
    },
    {
        "nombre": "Usuario con alto riesgo",
        "ingreso_mensual": 2200, "nivel_endeudamiento": 68, "frecuencia_ahorro": "Baja",
        "transacciones": [
            {"descripcion": "Arriendo Apartamento", "valor": 950},
            {"descripcion": "Factura CFE", "valor": 180},
            {"descripcion": "Uber", "valor": 220},
            {"descripcion": "Restaurante", "valor": 260},
            {"descripcion": "Pago minimo tarjeta de credito", "valor": 300},
        ],
    },
    {
        "nombre": "Usuario financieramente saludable",
        "ingreso_mensual": 8000, "nivel_endeudamiento": 8, "frecuencia_ahorro": "Alta",
        "transacciones": [
            {"descripcion": "Supermercado Carulla", "valor": 300},
            {"descripcion": "Farmacia Cruz Verde", "valor": 60},
            {"descripcion": "Netflix", "valor": 40},
        ],
    },
]

for ejemplo in ejemplos:
    resultado = analisis_financiero_completo(
        ejemplo["ingreso_mensual"], ejemplo["nivel_endeudamiento"],
        ejemplo["frecuencia_ahorro"], ejemplo["transacciones"],
    )
    print(f"=== {ejemplo['nombre']} ===")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
    print()

=== Ejemplo del brief ===
{
  "perfil_financiero": "Saludable",
  "probabilidad": 0.8611,
  "resumen_gastos": {
    "alimentacion": 460.0,
    "transporte": 300.0
  },
  "recomendaciones": [
    "Tu perfil financiero es saludable. Manten tus habitos actuales de gasto y ahorro.",
    "Podrias aumentar tu frecuencia de ahorro para fortalecer tu reserva financiera."
  ]
}

=== Usuario con alto riesgo ===
{
  "perfil_financiero": "En riesgo",
  "probabilidad": 0.9499,
  "resumen_gastos": {
    "transporte": 220.0,
    "vivienda": 950.0,
    "ocio": 560.0,
    "servicios": 180.0
  },
  "recomendaciones": [
    "Tu perfil financiero esta en riesgo. Te recomendamos revisar a fondo tus gastos y buscar reducir tu endeudamiento lo antes posible.",
    "Tu nivel de endeudamiento (68%) es alto; prioriza reducir el uso de credito y evita adquirir nuevas deudas.",
    "Tu gasto en ocio y entretenimiento representa el 25% de tu ingreso mensual (recomendado: hasta 10%); considera reducirlo o revisar e

## 6. Serializacion de las reglas para el equipo de backend

In [7]:
import os
os.makedirs("artefactos", exist_ok=True)

documentacion_reglas = {
    "reglas": REGLAS_RECOMENDACIONES,
    "nombres_categoria_display": NOMBRES_CATEGORIA,
    "mensajes_perfil_general": MENSAJES_PERFIL,
    "notas": (
        "Las recomendaciones se generan evaluando cada regla y asignandole una severidad "
        "(3=alta, 2=media, 1=informativa). Se ordenan por severidad descendente y se limitan "
        "a 'max_recomendaciones' (incluyendo siempre primero el mensaje general del perfil). "
        "Salud y Educacion no tienen umbral de alerta por categoria: gastar mas ahi no se "
        "considera necesariamente negativo."
    ),
}

with open("artefactos/reglas_recomendaciones.json", "w", encoding="utf-8") as f:
    json.dump(documentacion_reglas, f, ensure_ascii=False, indent=2)

print("Reglas guardadas en 'artefactos/reglas_recomendaciones.json'")
print("(el equipo de Back-End puede reimplementar esta misma logica en Java/Spring Boot")
print("sin ambiguedad, usando estos umbrales y plantillas de mensaje exactos)")

Reglas guardadas en 'artefactos/reglas_recomendaciones.json'
(el equipo de Back-End puede reimplementar esta misma logica en Java/Spring Boot
sin ambiguedad, usando estos umbrales y plantillas de mensaje exactos)


## Conclusiones

**El motor funciona como lo pide el brief**

Los tres ejemplos confirman que las recomendaciones cambian de forma coherente segun la situacion financiera real del usuario:

- El **usuario saludable** (endeudamiento 8%, ahorro alto, ratio gasto/ingreso ≈ 5%) recibio unicamente el mensaje general positivo — el motor no genera alertas falsas cuando no hay nada que alertar.
- El **usuario en riesgo** (endeudamiento 68%, ratio ≈ 87%, ahorro bajo) recibio 4 recomendaciones priorizadas por severidad: perfil general, endeudamiento alto (severidad 3), gasto en ocio 25% del ingreso (severidad 2) y ratio de gasto (severidad 2). La recomendacion de vivienda (severidad 1, ya que el 43% de gasto en esa categoria no superaba el 1.3x del umbral) quedo **descartada automaticamente** por el limite de `max_recomendaciones=4` — confirma que la priorizacion por severidad realmente filtra, no solo ordena.
- El **ejemplo del brief** (uso moderado) recibio el mensaje general mas una sugerencia leve de aumentar el ahorro (frecuencia "Media"), consistente con su perfil "Saludable" con probabilidad 86%.

**Dos observaciones honestas sobre el clasificador de gastos, reutilizado aqui**

1. La palabra "Streaming" sola (sin nombre de servicio como Netflix) tiene 0 coincidencias en el vocabulario TF-IDF y cae en Alimentacion por default — el mismo comportamiento ante vocabulario desconocido que documentamos con "Gimnasio" en el Paso 6. Es una limitacion consistente y ya conocida, no un caso nuevo.
2. "Restaurante" se clasifica como **Ocio**, no como Alimentacion. Esto no es un error del modelo: en nuestro catalogo sintetico, el unico comercio con la palabra "restaurante" es "Bar Restaurante Amigos", registrado bajo la categoria Ocio (Paso 1). El modelo aprendio exactamente la taxonomia que le dimos. Si el equipo considera que un restaurante deberia clasificarse como Alimentacion en vez de Ocio, es una decision de **diseno del catalogo de datos**, no del algoritmo — vale la pena revisarlo antes de la presentacion final.

**Artefactos guardados:** `artefactos/reglas_recomendaciones.json`, con los umbrales, nombres de categoria y plantillas de mensaje, para que el equipo de Back-End pueda reimplementar exactamente esta misma logica en Java/Spring Boot sin tener que adivinar los criterios.

**Pipeline de Ciencia de Datos completo:** con este notebook, el flujo end-to-end del brief (transacciones crudas -> clasificacion de gastos -> perfil financiero -> recomendaciones -> JSON de salida) queda implementado y probado con multiples ejemplos reales, listo para que el equipo de Back-End lo exponga como API REST.